# Chapter 34 — Transformers: Self-Attention

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch34/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# Self-attention: every position produces a query, a key, and a value.
# The query asks a question; the key advertises what a position holds;
# the value is what gets returned if that position is selected. This is
# Chapter 9's dot product, used to decide what to look at.
r = np.random.default_rng(34)
T, D = 5, 8                              # 5 positions, 8-dim embeddings
X = r.normal(size=(T, D))

Wq = r.normal(0, np.sqrt(1/D), (D, D))
Wk = r.normal(0, np.sqrt(1/D), (D, D))
Wv = r.normal(0, np.sqrt(1/D), (D, D))

Q = X @ Wq
K = X @ Wk
V = X @ Wv
print(f"X (input)   {X.shape}")
print(f"Q, K, V     {Q.shape}   one query, key, and value PER POSITION")

scores = Q @ K.T                          # every position scores every other
print(f"\nQ @ K.T     {scores.shape}   position i's query dotted "
      f"with every key")
print(f"scores[2]   {scores[2].round(2)}   <- position 2 vs all 5 keys")

weights = softmax(scores)
out = weights @ V
print(f"\nattention weights for position 2: {weights[2].round(3)}   sums to "
      f"{weights[2].sum():.4f}")
print(f"output for position 2 = weighted sum of all 5 value vectors:")
print(f"    {out[2, :4].round(3)}...")

X (input)   (5, 8)
Q, K, V     (5, 8)   one query, key, and value PER POSITION

Q @ K.T     (5, 5)   position i's query dotted with every key
scores[2]   [ 1.38  2.89  0.56 -1.56  1.78]   <- position 2 vs all 5 keys

attention weights for position 2: [0.133 0.603 0.059 0.007 0.198]   sums to 1.0000
output for position 2 = weighted sum of all 5 value vectors:
    [-0.133 -0.005 -0.686 -0.067]...


### Block 2  (`c2.py`)

In [4]:
# Why divide the scores by sqrt(d_k): the same variance-control argument
# as Chapter 31's initialization, applied to a dot product instead of a
# weighted sum. A dot product of two random d-dimensional vectors has
# variance proportional to d, so larger dimensions produce larger,
# more extreme scores before softmax ever sees them.
r2 = np.random.default_rng(34)

print(f"{'d_k':>6}{'score variance, unscaled':>26}"
      f"{'score variance, scaled':>25}")
for d in (8, 32, 128, 512):
    q = r2.normal(size=(2000, d))
    k = r2.normal(size=(2000, d))
    raw_scores = np.sum(q * k, axis=1)              # one dot product per row
    scaled_scores = raw_scores / np.sqrt(d)
    print(f"{d:>6}{raw_scores.var():>26.1f}{scaled_scores.var():>25.3f}")

print(f"\nunscaled variance grows linearly with d_k, exactly as an")
print(f"unscaled weighted sum's variance grew with fan-in in Chapter 31.")
print(f"dividing by sqrt(d_k) keeps it near 1 regardless of dimension.")

# real consequence for softmax: one query against five keys at d_k=64
d = 64
q = r2.normal(size=d)
K5 = r2.normal(size=(5, d))
raw = K5 @ q
print(f"\nfive real key vectors scored against one query, d_k = {d}:")
print(f"raw scores:       {raw.round(2)}")
print(f"unscaled softmax: {softmax(raw).round(4)}")
print(f"                  <- nearly one-hot")
print(f"scaled softmax:   {softmax(raw / np.sqrt(d)).round(4)}")
print(f"                  <- genuinely graded")

   d_k  score variance, unscaled   score variance, scaled
     8                       8.3                    1.037
    32                      31.4                    0.981
   128                     127.1                    0.993
   512                     501.8                    0.980

unscaled variance grows linearly with d_k, exactly as an
unscaled weighted sum's variance grew with fan-in in Chapter 31.
dividing by sqrt(d_k) keeps it near 1 regardless of dimension.

five real key vectors scored against one query, d_k = 64:
raw scores:       [ 9.78 17.58 14.82 -6.37  4.69]
unscaled softmax: [4.000e-04 9.402e-01 5.940e-02 0.000e+00 0.000e+00]
                  <- nearly one-hot
scaled softmax:   [0.1615 0.4283 0.3033 0.0214 0.0854]
                  <- genuinely graded


### Block 3  (`c3.py`)

In [5]:
# Self-attention has no built-in sense of order. Shuffle the input
# positions and the output shuffles identically: the network cannot
# tell "first" from "third" unless something tells it.
def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = Q.shape[-1]
    weights = softmax((Q @ K.T) / np.sqrt(d_k))
    return weights @ V

r3 = np.random.default_rng(34)
T, D = 5, 8
content = r3.normal(size=(T, D))          # what each position "contains"
Wq, Wk, Wv = (r3.normal(0, np.sqrt(1/D), (D, D)) for _ in range(3))

out = self_attention(content, Wq, Wk, Wv)

perm = [3, 0, 4, 1, 2]                    # shuffle which content sits where
out_shuffled_input = self_attention(content[perm], Wq, Wk, Wv)
out_then_shuffled = out[perm]

print(f"attend(shuffle(content))  row 0:")
print(f"    {out_shuffled_input[0].round(4)}")
print(f"shuffle(attend(content))  row 0:")
print(f"    {out_then_shuffled[0].round(4)}")
print(f"identical: {np.allclose(out_shuffled_input, out_then_shuffled)}")
print(f"\nself-attention commutes with any reordering of the input.")
print(f"it has no way to know which content came first without help.")

# a positional signal keyed to ARRAY INDEX, added fresh regardless of
# which content occupies that index -- this is the part a shuffle of
# the raw content cannot also shuffle away
idx = np.arange(T)[:, None] / T
pos_encoding = 0.8 * np.concatenate([np.sin(idx), np.cos(idx)] * (D // 2),
                                    axis=1)[:, :D]

def with_position(raw_content):
    return raw_content + pos_encoding     # position i always gets encoding i

out_pos = self_attention(with_position(content), Wq, Wk, Wv)
out_pos_shuffled_input = self_attention(with_position(content[perm]), Wq, Wk,
                                        Wv)
out_pos_then_shuffled = out_pos[perm]
print(f"\nwith positional encoding tied to array index:")
same = np.allclose(out_pos_shuffled_input, out_pos_then_shuffled, atol=1e-6)
print(f"identical: {same}")
print(f"max difference: "
      f"{np.abs(out_pos_shuffled_input - out_pos_then_shuffled).max():.4f}")

attend(shuffle(content))  row 0:
    [ 0.0649 -0.0642 -0.3058 -0.4307  0.0995 -0.0568  0.4099 -0.0652]
shuffle(attend(content))  row 0:
    [ 0.0649 -0.0642 -0.3058 -0.4307  0.0995 -0.0568  0.4099 -0.0652]
identical: True

self-attention commutes with any reordering of the input.
it has no way to know which content came first without help.

with positional encoding tied to array index:
identical: False
max difference: 0.1958


### Block 4  (`c4.py`)

In [6]:
# Multi-head attention runs several smaller attention operations in
# parallel, each with its own Q, K, V projections, then concatenates
# the results. Each head can specialize in a different kind of
# relationship between positions.
def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    T, D = X.shape
    d_h = D // n_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv                    # (T, D) each
    Qh = Q.reshape(T, n_heads, d_h).transpose(1, 0, 2)   # (heads, T, d_h)
    Kh = K.reshape(T, n_heads, d_h).transpose(1, 0, 2)
    Vh = V.reshape(T, n_heads, d_h).transpose(1, 0, 2)
    scores = Qh @ Kh.transpose(0, 2, 1) / np.sqrt(d_h)   # (heads, T, T)
    weights = softmax(scores)
    out_h = weights @ Vh                                 # (heads, T, d_h)
    concat = out_h.transpose(1, 0, 2).reshape(T, D)      # back to (T, D)
    return concat @ Wo, weights

r4 = np.random.default_rng(34)
T, D, n_heads = 5, 8, 2
X = r4.normal(size=(T, D))
Wq, Wk, Wv, Wo = (r4.normal(0, np.sqrt(1/D), (D, D)) for _ in range(4))

out, weights = multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads)
print(f"input            {X.shape}")
print(f"per-head Q/K/V   ({n_heads}, {T}, {D // n_heads})")
print(f"attention weights {weights.shape}   one {T}x{T} matrix per head")
print(f"output           {out.shape}   back to the original width")

print(f"\nhead 0 attention weights for position 2: {weights[0, 2].round(3)}")
print(f"head 1 attention weights for position 2: {weights[1, 2].round(3)}")
print(f"the two heads attend differently, from the same input.")

input            (5, 8)
per-head Q/K/V   (2, 5, 4)
attention weights (2, 5, 5)   one 5x5 matrix per head
output           (5, 8)   back to the original width

head 0 attention weights for position 2: [0.19  0.379 0.204 0.09  0.137]
head 1 attention weights for position 2: [0.205 0.22  0.127 0.1   0.348]
the two heads attend differently, from the same input.


### Block 5  (`c5.py`)

In [7]:
# The self-attention backward pass. Every position is simultaneously a
# query, a key, and a value for every other position, so a gradient
# arriving at the output must be routed back through all three roles.
# This is the last gradient this book derives, and it earns that title.
def attn_forward(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores)
    out = weights @ V
    return out, (X, Q, K, V, weights, d_k)

def attn_backward(dout, cache, Wq, Wk, Wv):
    X, Q, K, V, weights, d_k = cache
    dV = weights.T @ dout
    dweights = dout @ V.T
    dscores = weights * (dweights - (dweights * weights).sum(-1,
                                                             keepdims=True))
    dscores = dscores / np.sqrt(d_k)
    dQ = dscores @ K
    dK = dscores.T @ Q
    dWq = X.T @ dQ; dWk = X.T @ dK; dWv = X.T @ dV
    dX = dQ @ Wq.T + dK @ Wk.T + dV @ Wv.T
    return dX, dWq, dWk, dWv

r5 = np.random.default_rng(34)
T, D = 5, 8
X = r5.normal(size=(T, D))
Wq, Wk, Wv = (r5.normal(0, np.sqrt(1/D), (D, D)) for _ in range(3))

out, cache = attn_forward(X, Wq, Wk, Wv)
dout = r5.normal(size=out.shape)
dX, dWq, dWk, dWv = attn_backward(dout, cache, Wq, Wk, Wv)

def loss_fn(X_, Wq_, Wk_, Wv_):
    o, _ = attn_forward(X_, Wq_, Wk_, Wv_)
    return np.sum(o * dout)

eps = 1e-5
print(f"{'target':>12}{'analytic':>12}{'numerical':>12}{'match':>8}")
for name, arr, grad in [('Wq', Wq, dWq), ('Wk', Wk, dWk), ('Wv', Wv, dWv),
                        ('X', X, dX)]:
    flat = int(np.abs(grad).argmax())       # the largest-magnitude entry
    i, j = flat // grad.shape[1], flat % grad.shape[1]
    orig = arr[i, j]
    arr[i, j] = orig + eps; lp = loss_fn(X, Wq, Wk, Wv)
    arr[i, j] = orig - eps; lm = loss_fn(X, Wq, Wk, Wv)
    arr[i, j] = orig
    numeric = (lp - lm) / (2 * eps)
    match = abs(numeric - grad[i, j]) < 1e-4
    print(f"{name+str((i,j)):>12}{grad[i,j]:>12.6f}{numeric:>12.6f}"
          f"{str(match):>8}")

      target    analytic   numerical   match
    Wq(7, 2)    1.751294    1.751294    True
    Wk(2, 6)   -2.016464   -2.016464    True
    Wv(2, 3)   -5.368413   -5.368413    True
     X(2, 6)    1.945791    1.945791    True


### Block 6  (`c6.py`)

In [8]:
# A minimal transformer block: self-attention, a residual connection,
# then a small feedforward layer, on Chapter 33's exact recall task
# (the class is planted at position 0, however long the sequence runs).
def make_recall_task(n_samples, seq_len, seed):
    rr = np.random.default_rng(seed)
    first = rr.integers(0, 3, n_samples)
    seq = rr.normal(0, 0.3, (n_samples, seq_len, 3))
    seq[np.arange(n_samples), 0] = np.eye(3)[first]
    return seq, first

def batch_self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv                  # (n, T, D)
    d_k = Q.shape[-1]
    scores = np.einsum('ntd,nsd->nts', Q, K) / np.sqrt(d_k)
    weights = softmax(scores)
    out = np.einsum('nts,nsd->ntd', weights, V)
    return out, (X, Q, K, V, weights, d_k)

def train_transformer(seq_len, seed, epochs=150, D=16):
    Xs, ys = make_recall_task(600, seq_len, seed)
    Xs_te, ys_te = make_recall_task(200, seq_len, seed + 1)
    rr = np.random.default_rng(seed)
    D_in, D_out = 3, 3
    We = rr.normal(0, np.sqrt(1/D_in), (D_in, D))       # embed into D dims
    Wq, Wk, Wv = (rr.normal(0, np.sqrt(1/D), (D, D)) for _ in range(3))
    W1 = rr.normal(0, np.sqrt(2/D), (D, D)); b1 = np.zeros(D)     # tiny FFN
    Wo = rr.normal(0, np.sqrt(1/D), (D, D_out)); bo = np.zeros(D_out)
    Y = np.eye(3)[ys]
    eta = 0.3
    for _ in range(epochs):
        Xe = Xs @ We
        attn_out, cache = batch_self_attention(Xe, Wq, Wk, Wv)
        resid1 = Xe + attn_out
        ff = np.maximum(0, resid1 @ W1 + b1)
        resid2 = resid1 + ff
        pooled = resid2[:, -1]                       # last position's output
        p = softmax(pooled @ Wo + bo)

        dscore = (p - Y) / len(Xs)
        dWo = pooled.T @ dscore; dbo = dscore.sum(0)
        dpooled = dscore @ Wo.T
        dresid2 = np.zeros_like(resid2); dresid2[:, -1] = dpooled
        dff = dresid2 * (ff > 0)
        dW1 = resid1.reshape(-1, D).T @ dff.reshape(-1, D)
        db1 = dff.reshape(-1, D).sum(0)
        dresid1 = dresid2 + dff @ W1.T
        dattn = dresid1
        X_, Q, K, V, weights, d_k = cache
        dV = np.einsum('nts,ntd->nsd', weights, dattn)
        dweights = np.einsum('ntd,nsd->nts', dattn, V)
        dscores = weights * (dweights - (dweights * weights).sum(-1,
                             keepdims=True))
        dscores = dscores / np.sqrt(d_k)
        dQ = np.einsum('nts,nsd->ntd', dscores, K)
        dK = np.einsum('nts,ntd->nsd', dscores, Q)
        dWq = np.einsum('ntd,nte->de', X_, dQ)
        dWk = np.einsum('ntd,nte->de', X_, dK)
        dWv = np.einsum('ntd,nte->de', X_, dV)
        dXe = dresid1 + dQ @ Wq.T + dK @ Wk.T + dV @ Wv.T
        dWe = np.einsum('ntd,nte->de', Xs, dXe)

        Wo -= eta*dWo; bo -= eta*dbo; W1 -= eta*dW1; b1 -= eta*db1
        Wq -= eta*dWq; Wk -= eta*dWk; Wv -= eta*dWv; We -= eta*dWe

    Xe_te = Xs_te @ We
    attn_te, cache_te = batch_self_attention(Xe_te, Wq, Wk, Wv)
    weights_te = cache_te[4]
    # avg attention FROM the last position
    last_pos_weights = weights_te[:, -1, :].mean(0)
    r1_te = Xe_te + attn_te
    ff_te = np.maximum(0, r1_te @ W1 + b1)
    r2_te = r1_te + ff_te
    pred = softmax(r2_te[:, -1] @ Wo + bo).argmax(1)
    acc = (pred == ys_te).mean()
    return acc, last_pos_weights

print(f"{'length':>8}{'RNN+attn':>10}{'transformer':>13}"
      f"{'weight on pos 0':>18}"
      f"{'chance (1/T)':>14}")
# Chapter 33's attention column, quoted from that
# chapter's own run. It cannot be recomputed here: it
# belongs to a different session with a different model.
prior_attn = {2: 1.0, 5: 1.0, 10: 1.0, 20: 1.0, 40: 0.995}
for L in (2, 5, 10, 20, 40):
    acc, w = train_transformer(L, seed=34)
    print(f"{L:>8}{prior_attn[L]:>10.4f}{acc:>13.4f}"
          f"{w[0]:>18.4f}{1/L:>14.4f}")

  length  RNN+attn  transformer   weight on pos 0  chance (1/T)


       2    1.0000       1.0000            0.5556        0.5000


       5    1.0000       0.8950            0.2079        0.2000


      10    1.0000       0.7350            0.1072        0.1000


      20    1.0000       0.5600            0.0502        0.0500


      40    0.9950       0.4750            0.0252        0.0250
